# NB12 — MGnify MAG validation of the per-Mb metal homeostasis finding

Independent replication using two non-overlapping axes:
- **Genomes**: 260K MGnify MAGs (kescience_mgnify) — not kbase_ke_pangenome isolates
- **Niche breadth**: Shannon entropy of biome labels per genus — not arkinlab_microbeatlas Levins' B

**Primary finding to replicate**: Metal homeostasis gene density (KOs per Mb) predicts
ecological specialization, driven by Tier 2 homeostasis genes (p=0.011) not Tier 1
resistance genes (p=0.256).

**Blocks**:
- Block 0: Spark setup + schema check (JupyterHub)
- Block 1: Extract KO counts per MAG from kescience_mgnify.gene_eggnog (JupyterHub)
- Block 2: Biome-based niche breadth from mgnify_mag_metal_traits.csv (local)
- Block 3: Merge + PGLS via R subprocess (local)
- Block 4: Results summary and comparison table

In [1]:
# Block 0 — Imports and Spark setup
import sys, os, subprocess, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

sys.path.append('/opt/conda/lib/python3.13/site-packages')
from berdl_notebook_utils.setup_spark_session import get_spark_session
spark = get_spark_session()

from pyspark.sql import functions as F

DATA_DIR = '../data'
MIN_MAGS_PER_GENUS = 5   # minimum MAGs for stable niche estimate
KO_FILE  = os.path.join(DATA_DIR, 'mrg_ko_final.csv')
MAG_META = os.path.join(DATA_DIR, 'mgnify_mag_metal_traits.csv')
TREE     = os.path.join(DATA_DIR, 'gtdb_bac_genus_pruned.tree')
R_BIN    = '/home/hmacgregor/r_env/bin/Rscript'
SEED     = 42

print('Spark ready:', spark.version)

Spark ready: 4.0.1


In [2]:
# Block 0b — Schema check: find KEGG KO column in gene_eggnog
schema_df = spark.sql('DESCRIBE TABLE kescience_mgnify.gene_eggnog').toPandas()
print('gene_eggnog schema:')
print(schema_df[['col_name', 'data_type']].to_string(index=False))

# Auto-detect KEGG KO column (eggNOG-mapper uses KEGG_ko or kegg_ko)
col_lower = {c.lower(): c for c in schema_df['col_name'].tolist()}
candidates = ['kegg_ko', 'kegg_kos', 'ko', 'kegg_ko_id', 'kegg_ortholog']
KO_COL = next((col_lower[c] for c in candidates if c in col_lower), None)

if KO_COL:
    print(f"\nAuto-detected KEGG KO column: '{KO_COL}'")
else:
    print('\nWARNING: KEGG KO column not found. Set KO_COL manually.')
    print('Available columns:', schema_df['col_name'].tolist())
    # Fallback: use n_metal_types from mgnify_mag_metal_traits.csv
    KO_COL = None

gene_eggnog schema:
               col_name data_type
                gene_id    string
              genome_id    string
               biome_id    string
          seed_ortholog    string
                 evalue    double
                  score    double
             eggnog_ogs    string
          max_annot_lvl    string
           cog_category    string
            description    string
         preferred_name    string
               go_terms    string
                     ec    string
                kegg_ko    string
           kegg_pathway    string
            kegg_module    string
          kegg_reaction    string
                   cazy    string
                  pfams    string
# Partition Information          
             # col_name data_type
               biome_id    string

Auto-detected KEGG KO column: 'kegg_ko'


In [3]:
# Block 1 — Extract 94-KO counts per MAG from gene_eggnog (JupyterHub / Spark)
#
# eggNOG-mapper stores KEGG_ko as comma-separated values with a 'ko:' prefix,
# e.g. 'ko:K00363,ko:K00520'. We must explode and strip before joining.

assert KO_COL is not None, "KEGG KO column not found. Run the fallback cell instead."

ko_df    = pd.read_csv(KO_FILE)
ko_ids   = ko_df['ko_id'].tolist()
print(f'94-KO list: {len(ko_ids)} KOs  (Tier 1: {(ko_df.tier==1).sum()}, Tier 2: {(ko_df.tier==2).sum()})')

# Sample to confirm the kegg_ko format
sample = (spark.table('kescience_mgnify.gene_eggnog')
    .select(KO_COL)
    .filter(F.col(KO_COL).isNotNull() & (F.col(KO_COL) != ''))
    .limit(5)
    .toPandas())
print(f'\nSample {KO_COL} values:')
print(sample[KO_COL].tolist())

# Load quality MAGs list
mag_meta   = pd.read_csv(MAG_META)
genome_ids = mag_meta['genome_id'].tolist()
print(f'\nQuality MAGs: {len(genome_ids):,}')
genome_id_sdf = spark.createDataFrame([(g,) for g in genome_ids], ['genome_id'])

# KO tier lookup (keyed on bare KO id, e.g. 'K00363')
ko_tier_sdf = spark.createDataFrame(
    ko_df[['ko_id', 'tier']].rename(columns={'ko_id': 'ko_clean'})
)

# Explode comma-separated KO list and strip 'ko:' prefix
ko_exploded = (
    spark.table('kescience_mgnify.gene_eggnog')
    .join(genome_id_sdf, 'genome_id', 'inner')
    .filter(F.col(KO_COL).isNotNull() & (F.col(KO_COL) != ''))
    .withColumn('ko_split', F.explode(F.split(F.col(KO_COL), ',')))
    .withColumn('ko_clean', F.regexp_replace(F.trim(F.col('ko_split')), r'^ko:', ''))
    .filter(F.col('ko_clean') != '')
)

# Join with 94-KO tier table and count per genome
ko_counts_sdf = (
    ko_exploded
    .join(ko_tier_sdf, 'ko_clean', 'inner')
    .groupBy('genome_id')
    .agg(
        F.count('*').alias('n_ko_total'),
        F.sum(F.when(F.col('tier') == 1, 1).otherwise(0)).alias('n_ko_tier1'),
        F.sum(F.when(F.col('tier') == 2, 1).otherwise(0)).alias('n_ko_tier2'),
    )
)

print('\nCollecting KO counts...')
ko_pd = ko_counts_sdf.toPandas()
print(f'MAGs with >=1 metal KO: {len(ko_pd):,}  '
      f'(of {len(genome_ids):,} total; {len(ko_pd)/len(genome_ids):.1%})')

# Left-join to preserve all MAGs; fill 0 for those with no metal KOs
mag_ko = mag_meta.merge(ko_pd, on='genome_id', how='left')
mag_ko[['n_ko_total', 'n_ko_tier1', 'n_ko_tier2']] = (
    mag_ko[['n_ko_total', 'n_ko_tier1', 'n_ko_tier2']].fillna(0).astype(int)
)

# Normalize by genome size (length is in bp)
mag_ko['genome_length_mb'] = mag_ko['length'] / 1_000_000
for col in ['total', 'tier1', 'tier2']:
    mag_ko[f'ko_per_mb_{col}'] = (
        mag_ko[f'n_ko_{col}'] / mag_ko['genome_length_mb'].replace(0, np.nan)
    )

out1 = os.path.join(DATA_DIR, 'mgnify_mag_ko_density.csv')
mag_ko[['genome_id', 'lineage', 'genus', 'genome_length_mb',
        'n_ko_total', 'n_ko_tier1', 'n_ko_tier2',
        'ko_per_mb_total', 'ko_per_mb_tier1', 'ko_per_mb_tier2']].to_csv(out1, index=False)
print(f'Saved: {out1}  ({len(mag_ko):,} rows)')
print(mag_ko[['ko_per_mb_total', 'ko_per_mb_tier1', 'ko_per_mb_tier2']].describe().round(4))

94-KO list: 94 KOs  (Tier 1: 50, Tier 2: 44)



Sample kegg_ko values:
['ko:K03723', '-', 'ko:K03526', 'ko:K01425', 'ko:K07456']



Quality MAGs: 260,652


MAGs with >=1 metal KO: 25,669  (of 260,652 total; 9.8%)


Saved: ../data/mgnify_mag_ko_density.csv  (260,652 rows)
       ko_per_mb_total  ko_per_mb_tier1  ko_per_mb_tier2
count      260652.0000      260652.0000      260652.0000
mean            0.6650           0.3506           0.3143
std             2.3523           1.2898           1.1504
min             0.0000           0.0000           0.0000
25%             0.0000           0.0000           0.0000
50%             0.0000           0.0000           0.0000
75%             0.0000           0.0000           0.0000
max            43.4470          22.3449          29.2144


In [4]:
# Block 1 FALLBACK — Use n_metal_types from mgnify_mag_metal_traits.csv
# Run only if Block 1 above could not run (KO_COL is None).
#
# n_metal_types = number of distinct metal AMR classes per MAG (from gene_amr, not gene_eggnog).
# This is a coarser predictor than per-KO homeostasis counts, but already in the CSV.

if KO_COL is None:
    mag_meta = pd.read_csv(MAG_META)
    mag_ko   = mag_meta.copy()
    mag_ko['genome_length_mb']  = mag_ko['length'] / 1_000_000
    mag_ko['ko_per_mb_total']   = mag_ko['n_metal_types'] / mag_ko['genome_length_mb'].replace(0, np.nan)
    mag_ko['ko_per_mb_tier1']   = np.nan  # tier stratification unavailable in fallback
    mag_ko['ko_per_mb_tier2']   = np.nan
    out1 = os.path.join(DATA_DIR, 'mgnify_mag_ko_density.csv')
    mag_ko.to_csv(out1, index=False)
    print(f'FALLBACK: saved n_metal_types-based density to {out1}')
    print('NOTE: Tier-stratified PGLS unavailable without gene_eggnog KEGG KOs.')
else:
    print('gene_eggnog KO path succeeded; fallback not needed.')

gene_eggnog KO path succeeded; fallback not needed.


In [5]:
# Block 2 — Biome-based niche breadth per genus (local; no Spark needed)
#
# Uses mgnify_mag_metal_traits.csv which already has genus + biome_name per MAG.
# Niche breadth = normalized Shannon entropy of biome label distribution per genus.

mag_meta = pd.read_csv(MAG_META)
print(f'Loaded {len(mag_meta):,} MAGs; {mag_meta.genus.nunique()} unique genera')
print(f'Unique biomes: {mag_meta.biome_name.nunique()}')

# Count MAGs per genus × biome
genus_biome = (
    mag_meta.groupby(['genus', 'biome_name'])
    .size()
    .unstack(fill_value=0)
)

n_biomes     = genus_biome.shape[1]
proportions  = genus_biome.div(genus_biome.sum(axis=1), axis=0)
# Shannon H; log(0) avoided via small offset
biome_H      = -(proportions * np.log(proportions + 1e-300)).sum(axis=1)
# Normalize to [0,1]: divide by max possible H = log(n_biomes)
biome_H_std  = biome_H / np.log(n_biomes)

n_mags_per   = mag_meta.groupby('genus').size().rename('n_mags')
n_biomes_per = (genus_biome > 0).sum(axis=1).rename('n_biomes_occupied')

niche_df = pd.DataFrame({
    'n_mags':           n_mags_per,
    'n_biomes':         n_biomes_per,
    'biome_H':          biome_H,
    'biome_H_std':      biome_H_std,
}).reset_index()

# Filter: minimum MAGs per genus for stable entropy estimate
niche_filt = niche_df[niche_df['n_mags'] >= MIN_MAGS_PER_GENUS].copy()
print(f'Genera with >= {MIN_MAGS_PER_GENUS} MAGs: {len(niche_filt):,} '
      f'(dropped {len(niche_df)-len(niche_filt):,})')
print(niche_filt['biome_H_std'].describe().round(4))

out2 = os.path.join(DATA_DIR, 'mgnify_genus_biome_breadth.csv')
niche_filt.to_csv(out2, index=False)
print(f'Saved: {out2}')

Loaded 260,652 MAGs; 7541 unique genera
Unique biomes: 18
Genera with >= 5 MAGs: 2,164 (dropped 5,377)
count    2164.0000
mean        0.0943
std         0.1301
min        -0.0000
25%        -0.0000
50%        -0.0000
75%         0.1731
max         0.7033
Name: biome_H_std, dtype: float64
Saved: ../data/mgnify_genus_biome_breadth.csv


In [6]:
# Block 3 — Genus-level merge, z-score, and PGLS

from sklearn.preprocessing import StandardScaler

mag_ko_df = pd.read_csv(os.path.join(DATA_DIR, 'mgnify_mag_ko_density.csv'))
niche_df  = pd.read_csv(os.path.join(DATA_DIR, 'mgnify_genus_biome_breadth.csv'))

# ── Aggregate KO density to genus level ──────────────────────────────────────
genus_ko = (
    mag_ko_df
    .dropna(subset=['ko_per_mb_total'])
    .groupby('genus')[['ko_per_mb_total', 'ko_per_mb_tier1', 'ko_per_mb_tier2']]
    .mean()
    .reset_index()
)
print(f'Genera with KO density: {len(genus_ko):,}')

# ── Merge with niche breadth ──────────────────────────────────────────────────
merged = genus_ko.merge(niche_df[['genus', 'biome_H_std', 'n_mags']], on='genus', how='inner')
print(f'After inner merge (niche + KO density): {len(merged):,} genera')

# ── GTDB genus name normalisation ────────────────────────────────────────────
# MGnify lineage genus names may differ in case/prefix from GTDB tree tip labels.
# Convert to lower-case and strip GTDB 'g__' prefix if present.
merged['genus_lower'] = (
    merged['genus']
    .str.replace(r'^g__', '', regex=True)
    .str.lower()
    .str.strip()
)

# Deduplicate: multiple raw genus strings can normalize to the same genus_lower,
# causing duplicate rownames in R that break corPagel's tip-order matching.
n_before = len(merged)
merged = (merged
    .dropna(subset=['genus_lower'])
    .query("genus_lower != ''")
    .groupby('genus_lower', as_index=False)
    .agg({
        'biome_H_std':      'mean',
        'ko_per_mb_total':  'mean',
        'ko_per_mb_tier1':  'mean',
        'ko_per_mb_tier2':  'mean',
    })
)
if len(merged) < n_before:
    print(f'Deduplicated genus_lower: {n_before} → {len(merged)} genera')

# ── Z-score predictors ────────────────────────────────────────────────────────
pred_cols = ['ko_per_mb_total', 'ko_per_mb_tier1', 'ko_per_mb_tier2']
for col in pred_cols:
    vals = merged[col].values.reshape(-1, 1)
    if not np.all(np.isnan(vals)) and np.nanstd(vals) > 1e-10:
        merged[col + '_z'] = StandardScaler().fit_transform(vals).flatten()
    else:
        merged[col + '_z'] = np.nan
        print(f'WARNING: {col} has zero variance — z-score set to NaN (tier PGLS will be skipped)')

# ── Save PGLS input ───────────────────────────────────────────────────────────
pgls_cols = ['genus_lower', 'biome_H_std',
             'ko_per_mb_total_z', 'ko_per_mb_tier1_z', 'ko_per_mb_tier2_z']
pgls_input = merged[pgls_cols].dropna(subset=['biome_H_std', 'ko_per_mb_total_z'])
pgls_input_path  = os.path.join(DATA_DIR, 'mgnify_pgls_input.csv')
pgls_output_path = os.path.join(DATA_DIR, 'mgnify_validation_pgls.csv')
pgls_input.to_csv(pgls_input_path, index=False)
print(f'PGLS input saved: {pgls_input_path}  ({len(pgls_input)} genera)')

# ── Run PGLS via R subprocess ─────────────────────────────────────────────────
r_script = '../scripts/pgls_mgnify_validation.R'
cmd = [R_BIN, r_script, pgls_input_path, TREE, pgls_output_path]
print(f'\nRunning: {" ".join(cmd)}')
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError(f'R script failed (exit {result.returncode})')

Genera with KO density: 7,541
After inner merge (niche + KO density): 2,164 genera
PGLS input saved: ../data/mgnify_pgls_input.csv  (2164 genera)

Running: /home/hmacgregor/r_env/bin/Rscript ../scripts/pgls_mgnify_validation.R ../data/mgnify_pgls_input.csv ../data/gtdb_bac_genus_pruned.tree ../data/mgnify_validation_pgls.csv



=== MGnify MAG validation PGLS ===
Input:  ../data/mgnify_pgls_input.csv
Tree:   ../data/gtdb_bac_genus_pruned.tree
Output: ../data/mgnify_validation_pgls.csv

Loaded 2164 genera from input CSV
Tree has 2283 tips
After tree pruning: 576 genera

Predictors to test (3): ko_per_mb_total_z, ko_per_mb_tier1_z, ko_per_mb_tier2_z

[PGLS] biome_H_std ~ ko_per_mb_total_z  (n=576)
  lambda=0.3790  beta=0.0508  SE=0.0055  t=9.213  p=5.942e-19  deltaAIC=-77.28

[PGLS] biome_H_std ~ ko_per_mb_tier1_z  (n=576)
  lambda=0.3949  beta=0.0494  SE=0.0056  t=8.784  p=1.824e-17  deltaAIC=-70.59

[PGLS] biome_H_std ~ ko_per_mb_tier2_z  (n=576)
  lambda=0.3599  beta=0.0507  SE=0.0057  t=8.959  p=4.571e-18  deltaAIC=-73.04

Saved: ../data/mgnify_validation_pgls.csv  (3 models)

Summary:
               predictor n_taxa    lambda       beta      p_value delta_AIC
Value  ko_per_mb_total_z    576 0.3790495 0.05076211 5.942149e-19 -77.27628
Value1 ko_per_mb_tier1_z    576 0.3948776 0.04937147 1.823917e-17 -70.586

In [7]:
# Block 4 — Results summary and comparison with primary analysis

pgls_val = pd.read_csv(os.path.join(DATA_DIR, 'mgnify_validation_pgls.csv'))

# Primary analysis results (from pgls_results_tier_normalized.csv)
primary = {
    'ko_per_mb_total_z':  dict(beta=-0.022, p=4e-7,  n=997),
    'ko_per_mb_tier1_z':  dict(beta=-0.004, p=0.256, n=997),
    'ko_per_mb_tier2_z':  dict(beta=-0.009, p=0.011, n=997),
}

label_map = {
    'ko_per_mb_total_z':  'Total 94-KO / Mb',
    'ko_per_mb_tier1_z':  'Tier 1 resistance / Mb',
    'ko_per_mb_tier2_z':  'Tier 2 homeostasis / Mb',
}

print('='*80)
print('VALIDATION COMPARISON')
print(f'{"Predictor":<28}  {"Primary β":>10}  {"Primary p":>10}  {"Primary n":>9}  '
      f'{"MGnify β":>10}  {"MGnify p":>10}  {"MGnify n":>8}')
print('-'*80)
for _, row in pgls_val.iterrows():
    pred = row['predictor']
    p    = primary.get(pred, {})
    label = label_map.get(pred, pred)
    print(f'{label:<28}  {p.get("beta",float("nan")):>10.4f}  {p.get("p",float("nan")):>10.4g}  '
          f'{p.get("n",""):>9}  {row["beta"]:>10.4f}  {row["p_value"]:>10.4g}  {int(row["n_taxa"]):>8}')
print('='*80)

# Tier asymmetry check (primary success criterion)
t1_row = pgls_val[pgls_val['predictor'] == 'ko_per_mb_tier1_z']
t2_row = pgls_val[pgls_val['predictor'] == 'ko_per_mb_tier2_z']
if len(t1_row) and len(t2_row):
    t1_p = t1_row.iloc[0]['p_value']
    t2_p = t2_row.iloc[0]['p_value']
    t2_b = t2_row.iloc[0]['beta']
    tier_asymmetry = (t2_b < 0) and (t2_p < t1_p)
    print(f'\nTier asymmetry replicated: {tier_asymmetry} '
          f'(Tier2 β={t2_b:.4f}, p={t2_p:.4g} vs Tier1 p={t1_p:.4g})')
else:
    print('\nTier-stratified PGLS not available (fallback mode or convergence failure).')

VALIDATION COMPARISON
Predictor                      Primary β   Primary p  Primary n    MGnify β    MGnify p  MGnify n
--------------------------------------------------------------------------------
Total 94-KO / Mb                 -0.0220       4e-07        997      0.0508   5.942e-19       576
Tier 1 resistance / Mb           -0.0040       0.256        997      0.0494   1.824e-17       576
Tier 2 homeostasis / Mb          -0.0090       0.011        997      0.0507   4.571e-18       576

Tier asymmetry replicated: False (Tier2 β=0.0507, p=4.571e-18 vs Tier1 p=1.824e-17)


---
## Sensitivity analysis: FitnessBrowser empirical KO list

Repeat the MGnify MAG PGLS using the **74 empirically derived FitnessBrowser KOs**
(`fitnessbrowser_metal_gene_list/data/fitnessbrowser_metal_kos_unique.csv`) instead of
the curated 94-KO list. These KOs were identified from RB-TnSeq fitness data: genes with
`min_fit < −1.0` and `|t| > 4` under metal stress, appearing in ≥2 organisms.

All 74 KOs are Tier 1 (resistance metals only). Tier 2 will have zero variance and be auto-skipped by the R PGLS script.

**Expected result if signal is robust:** same sign (β > 0) and similar magnitude to the
curated-list result (β=+0.051, p=5.9×10⁻¹⁹, n=576). A weaker but concordant result
confirms the association is not driven by literature over-representation of particular gene families.


In [8]:

# Block 5 — FitnessBrowser 74-KO sensitivity analysis (Spark query + PGLS)
#
# Identical pipeline to Block 1+3 but with the empirically derived FitnessBrowser KO list.
# All 74 KOs are Tier 1; Tier 2 will be zero-variance and auto-skipped by the R script.

from sklearn.preprocessing import StandardScaler

FB_KO_FILE = os.path.join(DATA_DIR,
    '../../fitnessbrowser_metal_gene_list/data/fitnessbrowser_metal_kos_unique.csv')
assert os.path.exists(FB_KO_FILE), f'Run fitnessbrowser_metal_gene_list NB01-NB03 first: {FB_KO_FILE}'

fb_ko_df = pd.read_csv(FB_KO_FILE)
fb_ko_ids = fb_ko_df['ko_id'].tolist()
print(f'FitnessBrowser KO list: {len(fb_ko_ids)} KOs  '
      f'(Tier 1: {(fb_ko_df.tier==1).sum()}, Tier 2: {(fb_ko_df.tier==2).sum()})')

# Load MAG metadata (same as Block 1)
mag_meta   = pd.read_csv(MAG_META)
genome_ids = mag_meta['genome_id'].tolist()
genome_id_sdf = spark.createDataFrame([(g,) for g in genome_ids], ['genome_id'])

# KO lookup table — all Tier 1
fb_tier_sdf = spark.createDataFrame(
    fb_ko_df[['ko_id']].assign(tier=1).rename(columns={'ko_id': 'ko_clean'})
)

# Explode and filter to FB KOs
fb_ko_exploded = (
    spark.table('kescience_mgnify.gene_eggnog')
    .join(genome_id_sdf, 'genome_id', 'inner')
    .filter(F.col(KO_COL).isNotNull() & (F.col(KO_COL) != ''))
    .withColumn('ko_split', F.explode(F.split(F.col(KO_COL), ',')))
    .withColumn('ko_clean', F.regexp_replace(F.trim(F.col('ko_split')), r'^ko:', ''))
    .filter(F.col('ko_clean') != '')
)

fb_counts_sdf = (
    fb_ko_exploded
    .join(fb_tier_sdf, 'ko_clean', 'inner')
    .groupBy('genome_id')
    .agg(F.count('*').alias('n_ko_fb_total'))
)

print('Collecting FitnessBrowser KO counts...')
fb_pd = fb_counts_sdf.toPandas()
print(f'MAGs with >=1 FB KO: {len(fb_pd):,}  '
      f'(of {len(genome_ids):,} total; {len(fb_pd)/len(genome_ids):.1%})')

# Merge and normalize
fb_mag_ko = mag_meta.merge(fb_pd, on='genome_id', how='left')
fb_mag_ko['n_ko_fb_total'] = fb_mag_ko['n_ko_fb_total'].fillna(0).astype(int)
fb_mag_ko['genome_length_mb'] = fb_mag_ko['length'] / 1_000_000
fb_mag_ko['fb_ko_per_mb'] = (
    fb_mag_ko['n_ko_fb_total'] / fb_mag_ko['genome_length_mb'].replace(0, float('nan'))
)

out_fb = os.path.join(DATA_DIR, 'fb_mag_ko_density.csv')
fb_mag_ko[['genome_id', 'lineage', 'genus', 'genome_length_mb',
           'n_ko_fb_total', 'fb_ko_per_mb']].to_csv(out_fb, index=False)
print(f'Saved: {out_fb}  ({len(fb_mag_ko):,} rows)')
print(fb_mag_ko['fb_ko_per_mb'].describe().round(4))

# ── Aggregate to genus + merge with niche breadth ────────────────────────────
niche_df = pd.read_csv(os.path.join(DATA_DIR, 'mgnify_genus_biome_breadth.csv'))

fb_genus_ko = (
    fb_mag_ko.dropna(subset=['fb_ko_per_mb'])
    .groupby('genus')['fb_ko_per_mb'].mean().reset_index()
)
fb_merged = fb_genus_ko.merge(
    niche_df[['genus', 'biome_H_std']], on='genus', how='inner'
)
fb_merged['genus_lower'] = (
    fb_merged['genus'].str.replace(r'^g__', '', regex=True).str.lower().str.strip()
)
fb_merged = (fb_merged
    .dropna(subset=['genus_lower'])
    .query("genus_lower != ''")
    .groupby('genus_lower', as_index=False)
    .agg({'biome_H_std': 'mean', 'fb_ko_per_mb': 'mean'})
)
print(f'\nGenera for FB sensitivity PGLS: {len(fb_merged):,}')

# Z-score
fb_vals = fb_merged['fb_ko_per_mb'].values.reshape(-1, 1)
fb_merged['fb_ko_per_mb_z'] = StandardScaler().fit_transform(fb_vals).flatten()

# Save PGLS input and run R
fb_pgls_input  = os.path.join(DATA_DIR, 'fb_sensitivity_pgls_input.csv')
fb_pgls_output = os.path.join(DATA_DIR, 'fb_sensitivity_pgls.csv')
fb_merged[['genus_lower', 'biome_H_std', 'fb_ko_per_mb_z']].to_csv(fb_pgls_input, index=False)

r_script = '../scripts/pgls_mgnify_validation.R'
cmd = [R_BIN, r_script, fb_pgls_input, TREE, fb_pgls_output]
print(f'\nRunning PGLS: {" ".join(cmd)}')
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError(f'R script failed (exit {result.returncode})')


FitnessBrowser KO list: 74 KOs  (Tier 1: 74, Tier 2: 0)


MAGs with >=1 FB KO: 25,705  (of 260,652 total; 9.9%)


Saved: ../data/fb_mag_ko_density.csv  (260,652 rows)
count    260652.000
mean          2.329
std           7.610
min           0.000
25%           0.000
50%           0.000
75%           0.000
max          96.733
Name: fb_ko_per_mb, dtype: float64

Genera for FB sensitivity PGLS: 2,164

Running PGLS: /home/hmacgregor/r_env/bin/Rscript ../scripts/pgls_mgnify_validation.R ../data/fb_sensitivity_pgls_input.csv ../data/gtdb_bac_genus_pruned.tree ../data/fb_sensitivity_pgls.csv



=== MGnify MAG validation PGLS ===
Input:  ../data/fb_sensitivity_pgls_input.csv
Tree:   ../data/gtdb_bac_genus_pruned.tree
Output: ../data/fb_sensitivity_pgls.csv

Loaded 2164 genera from input CSV
Tree has 2283 tips
After tree pruning: 576 genera

Predictors to test (1): fb_ko_per_mb_z

[PGLS] biome_H_std ~ fb_ko_per_mb_z  (n=576)
  lambda=0.3744  beta=0.0530  SE=0.0061  t=8.654  p=5.027e-17  deltaAIC=-68.44

Saved: ../data/fb_sensitivity_pgls.csv  (1 models)

Summary:
           predictor n_taxa    lambda       beta      p_value delta_AIC
Value fb_ko_per_mb_z    576 0.3744023 0.05300114 5.027422e-17 -68.43743



In [9]:

# Block 6 — Sensitivity comparison: curated 94-KO vs FitnessBrowser 74-KO

fb_result = pd.read_csv(os.path.join(DATA_DIR, 'fb_sensitivity_pgls.csv'))
curated_result = pd.read_csv(os.path.join(DATA_DIR, 'mgnify_validation_pgls.csv'))

fb_row = fb_result.iloc[0]
cur_row = curated_result[curated_result['predictor'] == 'ko_per_mb_total_z'].iloc[0]

print('='*70)
print('GENE LIST SENSITIVITY ANALYSIS — MGnify MAG PGLS')
print('Response: biome_H_std (Shannon entropy of biome distribution per genus)')
print('='*70)
print(f'{"Gene list":<30}  {"n_KOs":>6}  {"n_genera":>9}  {"β":>8}  {"p":>10}  {"λ":>6}')
print('-'*70)
print(f'{"Curated (94 KOs)":<30}  {94:>6}  {int(cur_row.n_taxa):>9}  '
      f'{cur_row.beta:>8.4f}  {cur_row.p_value:>10.4g}  {cur_row.lambda_:>6.3f}' 
      if 'lambda_' in cur_row.index else
      f'{"Curated (94 KOs)":<30}  {94:>6}  {int(cur_row.n_taxa):>9}  '
      f'{cur_row.beta:>8.4f}  {cur_row.p_value:>10.4g}  {cur_row["lambda"]:>6.3f}')
print(f'{"FitnessBrowser (74 KOs)":<30}  {74:>6}  {int(fb_row.n_taxa):>9}  '
      f'{fb_row.beta:>8.4f}  {fb_row.p_value:>10.4g}  {fb_row["lambda"]:>6.3f}')
print('='*70)

# Concordance check
same_sign  = (cur_row.beta > 0) == (fb_row.beta > 0)
fb_sig     = fb_row.p_value < 0.05
print(f'\nSame sign:       {same_sign}')
print(f'FB significant:  {fb_sig} (p={fb_row.p_value:.4g})')
if same_sign and fb_sig:
    print('\n→ RESULT: Signal replicates with empirically derived gene list.')
    print('  The association is not dependent on literature-curated gene selection.')
elif same_sign and not fb_sig:
    print('\n→ RESULT: Same direction but weaker signal with empirical list.')
    print('  Consistent with fewer KOs (74 vs 94) and narrower metal scope (no Tier 2).')
else:
    print('\n→ RESULT: Sign reversal — interpret with caution.')
    print('  Check n_genera overlap and whether FB KOs have sufficient MAG coverage.')


GENE LIST SENSITIVITY ANALYSIS — MGnify MAG PGLS
Response: biome_H_std (Shannon entropy of biome distribution per genus)
Gene list                        n_KOs   n_genera         β           p       λ
----------------------------------------------------------------------
Curated (94 KOs)                    94        576    0.0508   5.942e-19   0.379
FitnessBrowser (74 KOs)             74        576    0.0530   5.027e-17   0.374

Same sign:       True
FB significant:  True (p=5.027e-17)

→ RESULT: Signal replicates with empirically derived gene list.
  The association is not dependent on literature-curated gene selection.
